In [7]:
from langsmith import Client
import pandas as pd
from dotenv import load_dotenv


DATASET_NAME="financebench-dev"

load_dotenv()

client=Client()

df_sample_finance=pd.read_json("../data/processed/financebench_p.json",lines=True)
df_sample_dev=df_sample_finance[df_sample_finance["split"]=="dev"]
    
examples=[]
for _, row in df_sample_dev.iterrows():
    evidence_pages=[[ev.get("doc_name") or ev.get("evidence_doc_name"), int(ev.get("evidence_page_num"))] for ev in row["evidence"]]
    examples.append({
        "inputs": {"question": row["question"]},
        "outputs": {"answer": row["answer"],"evidence_pages":evidence_pages },
        "metadata": {"financebench_id": row["financebench_id"],
                     "question_type": row["question_type"]}
    })   

print(examples[0])          

{'inputs': {'question': 'What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.'}, 'outputs': {'answer': '$1577.00', 'evidence_pages': [['3M_2018_10K', 59]]}, 'metadata': {'financebench_id': 'financebench_id_03029', 'question_type': 'metrics-generated'}}


In [ ]:
if client.has_dataset(dataset_name=DATASET_NAME):
    print("Il dataset esiste già, non lo ricreo.")
else:
    dataset=client.create_dataset(
        dataset_name=DATASET_NAME,
        description="FinanceBench open-source development split (51 questions)"
        )

    client.create_examples(dataset_id=dataset.id, examples=examples)
        